In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dataset1/sample_submission.csv
/kaggle/input/dataset1/train.csv
/kaggle/input/dataset1/test.csv


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from mlxtend.plotting import plot_decision_regions

# 1. Load the Data sets

In [ ]:
train_data=pd.read_csv('/kaggle/input/dataset1/train.csv')
test_data=pd.read_csv('/kaggle/input/dataset1/test.csv')

# 2. Data Encoding and Feautre/Target split
* Here, I applied a lamda function to dummy code the dicatimous labels "M" = 1
* Next, I seprated the features and targets in the trainining data

In [ ]:
# Encoded the target variable
train_data['label'] = train_data['label'].apply(lambda x: 1 if x == 'M' else 0)

# Separate features and target
X = train_data.drop(['id', 'label'], axis=1)
y = train_data['label']

#3. Splitting Training Data/Feature Scaling
* I used 80% training and 20% testing and the ramdom_state was set to 42
* I also started off with a Loegestic Regression and condituned on to other models accessing accuracy at each instance

In [ ]:
# Split the training data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardized the features
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_val_std = scaler.transform(X_val)

# Initiate the logistic regression model
lr = LogisticRegression(solver='lbfgs', multi_class='ovr')
lr.fit(X_train_std, y_train)

# Make predictions on the test set
y_pred = lr.predict(X_val_std)

# Calculate the accuracy on the test set
accuracy = accuracy_score(y_val, y_pred)
print(f'Logistic Regression accuracy on test set: {accuracy * 100:.2f}%')



Logistic Regression accuracy on test set: 97.80%


In [ ]:
# Instantiate and train the SVC model
svc = SVC(kernel='rbf', C=10, gamma='scale')  # Use RBF kernel, adjust C and gamma as needed
svc.fit(X_train_std, y_train)

# Make predictions on the test set
y_pred = svc.predict(X_val_std)

# Calculate the accuracy on the test set
accuracy = accuracy_score(y_val, y_pred)
print(f'SVC accuracy on test set: {accuracy * 100:.2f}%')


SVC accuracy on test set: 98.90%


# 4. Hyperparameter tuning for the SVC
* Used GridSearchCV to evalaute the SVC perfromance on the validation set and applied Principal Comment Analysis(PCA) for dimensionality reduction
* Next, I trained on the full set, however was not happy with the output

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, cross_val_score
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# GridSearchCV for hyperparameter tunning
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 1],
    'kernel': ['rbf', 'linear', 'poly', 'sigmoid']
}

# SVC Initialization
svc = SVC()

# GridSearchCV
grid_search = GridSearchCV(svc, param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(X_train_std, y_train)

# Best parameters and estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
print(f'Best SVC parameters: {best_params}')

# Predictions on the validation set
y_pred = best_model.predict(X_val_std)
accuracy = accuracy_score(y_val, y_pred)
print(f'Best SVC accuracy on validation set: {accuracy * 100:.2f}%')

# PCA for dimensionality reduction
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_std)
X_val_pca = pca.transform(X_val_std)

# Re-trained the SVC model with transformed data
best_model.fit(X_train_pca, y_train)
y_pred_pca = best_model.predict(X_val_pca)
accuracy_pca = accuracy_score(y_val, y_pred_pca)
print(f'SVC accuracy with PCA on validation set: {accuracy_pca * 100:.2f}%')

# Aassesed cross-validation
cv_scores = cross_val_score(best_model, X_train_std, y_train, cv=5, scoring='accuracy')
print(f'Cross-validated accuracy: {np.mean(cv_scores) * 100:.2f}%')

# Training on the full dataset
X_train_np = np.array(X)
y_train_np = np.array(y)

# Fit model on the full dataset
best_model.fit(X_train_np, y_train_np)

# Predictions on the full set
y_pred_full = best_model.predict(X_train_np)

# Accuracy calculations on the full set
accuracy_full = accuracy_score(y_train_np, y_pred_full)
print(f'SVC accuracy on full training set: {accuracy_full * 100:.2f}%')



Fitting 5 folds for each of 64 candidates, totalling 320 fits
Best SVC parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best SVC accuracy on validation set: 98.90%
SVC accuracy with PCA on validation set: 93.41%
Cross-validated accuracy: 98.07%
SVC accuracy on full training set: 96.26%


# 5. Training Continued
* Trained a perceptron, accuracy was 97.80%
* Trained a Random Forest, accuracy was 97.80%
* Trained a KNN and when I changed the n_nieghbors hyperparameter to 10 and got a 100% accuracy, however when I gengralized to the full set it was significantly lower at 95.38%

In [ ]:
from sklearn.linear_model import Perceptron

# Instantiated and trained the Perceptron model
perceptron = Perceptron(max_iter=1000, eta0=1, random_state=42)
perceptron.fit(X_train_std, y_train)

# Predictions on the validation set
y_pred = perceptron.predict(X_val_std)

# Calculate the accuracy on the validation set
accuracy = accuracy_score(y_val, y_pred)
print(f'Perceptron accuracy on validation set: {accuracy * 100:.2f}%')

Perceptron accuracy on validation set: 97.80%


In [ ]:
# Instantiate and train the Random Forest model
rf = RandomForestClassifier(n_estimators=100, random_state=42)  # You can adjust n_estimators and other hyperparameters
rf.fit(X_train_std, y_train)

# Make predictions on the validation set
y_pred = rf.predict(X_val_std)

# Calculate the accuracy on the validation set
accuracy = accuracy_score(y_val, y_pred)
print(f'Accuracy on validation set: {accuracy * 100:.2f}%')

Accuracy on validation set: 96.70%


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# Instantiate and train the KNN model
knn = KNeighborsClassifier(n_neighbors=10)  # Adjusted the n_neihbors and got a 100%
knn.fit(X_train_std, y_train)

# Make predictions on the validation set
y_pred = knn.predict(X_val_std)

# Calculate the accuracy on the validation set
accuracy = accuracy_score(y_val, y_pred)
print(f'KNN accuracy on test set: {accuracy * 100:.2f}%')

KNN accuracy on test set: 100.00%


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.preprocessing import StandardScaler

# Hyperparameter tuning using GridSearchCV
param_grid = {
    'n_neighbors': [3, 5, 10, 15, 20],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1, 2]
}

# Initialized the KNN model
knn = KNeighborsClassifier()

# Performed GridSearchCV using cross-validation
grid_search = GridSearchCV(knn, param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(X_train_std, y_train)

# Best parameters and estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
print(f'Best KNN parameters: {best_params}')

# Predictions on the test set
y_pred = best_model.predict(X_val_std)
accuracy = accuracy_score(y_val, y_pred)
print(f'Best KNN accuracy on validation set: {accuracy * 100:.2f}%')

# Cross-validation to assess generalization performance
cv_scores = cross_val_score(best_model, X_train_std, y_train, cv=5, scoring='accuracy')
print(f'Cross-validated accuracy: {np.mean(cv_scores) * 100:.2f}%')

# Training on the full dataset:
X_train_np = np.array(X)
y_train_np = np.array(y)

# Train the model on the full dataset
best_model.fit(X_train_np, y_train_np)

# Make predictions
y_pred_full = best_model.predict(X_train_np)

# Accuracy on the full training set
accuracy_full = accuracy_score(y_train_np, y_pred_full)
print(f'Best KNN accuracy on full training set: {accuracy_full * 100:.2f}%')




Fitting 5 folds for each of 80 candidates, totalling 400 fits
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.0s
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.0s
[CV] END .................C=0.1, gamma=scale, kernel=sigmoid; total time=   0.0s
[CV] END .................C=0.1, gamma=scale, kernel=sigmoid; total time=   0.0s
[CV] END ......................C=0.1, gamma=auto, kernel=rbf; total time=   0.0s
[CV] END ...................C=0.1, gamma=auto, kernel=linear; total time=   0.0s
[CV] END ...................C=0.1, gamma=auto, kernel=linear; total time=   0.0s
[CV] END ...................C=0.1, gamma=auto, 

#6. One last attempt with the SVM using the Sigmoid Kernel
* The first SVM had an accruacy of 98.90% but I thought I could imporve it

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, cross_val_score
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# GridSearchCV for hyperparameter tunning
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 1],
    'kernel': ['rbf', 'linear', 'poly', 'sigmoid']
}

# SVC Initialization
svc = SVC()

# GridSearchCV
grid_search = GridSearchCV(svc, param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(X_train_std, y_train)

# Best parameters and estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
print(f'Best SVC parameters: {best_params}')

# Predictions on the validation set
y_pred = best_model.predict(X_val_std)
accuracy = accuracy_score(y_val, y_pred)
print(f'Best SVC accuracy on validation set: {accuracy * 100:.2f}%')

# PCA for dimensionality reduction
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_std)
X_val_pca = pca.transform(X_val_std)

# Re-trained the SVC model with transformed data
best_model.fit(X_train_pca, y_train)
y_pred_pca = best_model.predict(X_val_pca)
accuracy_pca = accuracy_score(y_val, y_pred_pca)
print(f'SVC accuracy with PCA on validation set: {accuracy_pca * 100:.2f}%')

# Aassesed cross-validation
cv_scores = cross_val_score(best_model, X_train_std, y_train, cv=5, scoring='accuracy')
print(f'Cross-validated accuracy: {np.mean(cv_scores) * 100:.2f}%')

# Training on the full dataset
X_train_np = np.array(X)
y_train_np = np.array(y)

# Fit model on the full dataset
best_model.fit(X_train_np, y_train_np)

# Predictions on the full set
y_pred_full = best_model.predict(X_train_np)

# Accuracy calculations on the full set
accuracy_full = accuracy_score(y_train_np, y_pred_full)
print(f'SVC accuracy on full training set: {accuracy_full * 100:.2f}%')



Fitting 5 folds for each of 64 candidates, totalling 320 fits
Best SVC parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best SVC accuracy on validation set: 98.90%
SVC accuracy with PCA on validation set: 93.41%
Cross-validated accuracy: 98.07%
SVC accuracy on full training set: 96.26%


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Training the SVC model with the sigmoid kernel
svc = SVC(kernel='sigmoid', C=1.0, gamma='scale')
svc.fit(X_train_std, y_train)

# Predictions on the validation set
y_pred = svc.predict(X_val_std)

# Calculate the accuracy on the validation set
accuracy = accuracy_score(y_val, y_pred)
print(f'SVC accuracy on validation set with sigmoid kernel: {accuracy * 100:.2f}%')


SVC accuracy on validation set with sigmoid kernel: 97.80%
